In [ ]:
import pandas as pd


In [ ]:
# prompt: read the file I just uploaded, one is data kinase.xlsx

import pandas as pd
df = pd.read_excel('data kinase.xlsx')
print(df.head())


  Uniprotid kinase  label  Prediction
0    P30307   PLK2      0           1
1    P35548    KIT      0           0
2    Q9QXJ1   ABL2      1           0
3    P15822   CHK1      1           1
4    Q9Y5S9  SRPK1      1           1


In [ ]:

first_row = df.iloc[0]

# Get the values from 'Uniprot ID' and 'Kinase' columns
uniprot_id = first_row['Uniprotid']
kinase = first_row['kinase']

print(f"Uniprot ID of the first row: {uniprot_id}")
print(f"Kinase of the first row: {kinase}")


Uniprot ID of the first row: P30307
Kinase of the first row: PLK2


In [ ]:
# prompt: read the all_database_info.json

import json

with open('all_database_info.json', 'r') as f:
  database_info = json.load(f)


In [ ]:
# prompt: how to get the value from the json? I know the key that namse is "P30307" print the value

# Access the value associated with the key "P30307"
if "P30307" in database_info:
  p30307_value = database_info["P30307"]
  print(p30307_value)
else:
  print("Key 'P30307' not found in the JSON data.")

{'Protein_name': 'M-phase inducer phosphatase 3', 'Gene': 'CDC25C', 'Function': 'Functions as a dosage-dependent inducer in mitotic control. Tyrosine protein phosphatase required for progression of the cell cycle (PubMed:8119945). When phosphorylated, highly effective in activating G2 cells into prophase (PubMed:8119945). Directly dephosphorylates CDK1 and activates its kinase activity (PubMed:8119945)'}


In [ ]:
formatted_prompts = []
for i in range(2219) :
  row = df.iloc[i]
  uniprot_id = row['Uniprotid']
  kinase = row['kinase']
  value_of_uniprot_id = database_info[uniprot_id]
  value_of_kinase = database_info[kinase]
  prompt = [
    {"role": "system", "content": (
        "You are a bioinformatics expert evaluating potential phosphorylation interactions between kinases and substrates. "
        "Based solely on the provided descriptions of both proteins, assess whether the kinase is likely to phosphorylate the substrate. "
        "If there is any evidence or functional relevance suggesting potential phosphorylation, respond with 'Yes'. "
        "Otherwise, respond with 'No'. Do not provide explanations. Only answer 'Yes' or 'No'."
    )},
        {
            "role": "user",
            "content": (
                f"Protein A (Kinase): {kinase}\n"
                f"Protein B (Substrate): {uniprot_id}\n\n"
                f"Protein A Description: {value_of_kinase['Function']}\n"
                f"Protein B Description: {value_of_uniprot_id['Function']}\n"
            )
        }
  ]
  formatted_prompts.append(prompt)

In [ ]:
#——————————————————————————————————————————————————————————————————————————————————————————

formatted_prompts = []
for i in range(2219) :
  row = df.iloc[i]
  uniprot_id = row['Uniprotid']
  kinase = row['kinase']
  value_of_uniprot_id = database_info[uniprot_id]
  value_of_kinase = database_info[kinase]
  prompt = [
        {
            "role": "system",
            "content": (
                "You are a bioinformatics expert specializing in protein phosphorylation. "
                "You are asked to assess kinase-substrate relationships based solely on the provided evidence. "
                "Read the descriptions carefully and answer only 'Yes' or 'No'. Do not provide explanations or additional text, regardless of your confidence."
          )
        },
        {
            "role": "user",
            "content": (
                f"Protein A (Kinase): {kinase}\n"
                f"Protein B (Substrate): {uniprot_id}\n\n"
                f"Protein A Description: {value_of_kinase['Function']}\n"
                f"Protein B Description: {value_of_uniprot_id['Function']}\n"
            )
        }
  ]
  formatted_prompts.append(prompt)

  # 原始款舞动

In [ ]:
# prompt: 打印formatted_prompts[34]并且要看到完整的格式

import json

print(json.dumps(formatted_prompts[10], indent=2))
# You can also pretty print directly if you prefer, without importing json first
# from pprint import pprint
# pprint(formatted_prompts[34])

[
  {
    "role": "system",
    "content": "You are a bioinformatics expert evaluating potential phosphorylation interactions between kinases and substrates. Based solely on the provided descriptions of both proteins, assess whether the kinase is likely to phosphorylate the substrate. If there is any evidence or functional relevance suggesting potential phosphorylation, respond with 'Yes'. Otherwise, respond with 'No'. Do not provide explanations. Only answer 'Yes' or 'No'."
  },
  {
    "role": "user",
    "content": "Protein A (Kinase): CDK4\nProtein B (Substrate): Q9UDY8\n\nProtein A Description: Ser/Thr-kinase component of cyclin D-CDK4 (DC) complexes that phosphorylate and inhibit members of the retinoblastoma (RB) protein family including RB1 and regulate the cell-cycle during G(1)/S transition. Phosphorylation of RB1 allows dissociation of the transcription factor E2F from the RB/E2F complexes and the subsequent transcription of E2F target genes which are responsible for the pro

In [ ]:
print(formatted_prompts[34])

[{'role': 'system', 'content': "You are a bioinformatics expert evaluating potential phosphorylation interactions between kinases and substrates. Based solely on the provided descriptions of both proteins, assess whether the kinase is likely to phosphorylate the substrate. If there is any evidence or functional relevance suggesting potential phosphorylation, respond with 'Yes'. Otherwise, respond with 'No'. Do not provide explanations. Only answer 'Yes' or 'No'."}, {'role': 'user', 'content': "Protein A (Kinase): MST2\nProtein B (Substrate): P51955\n\nProtein A Description: Stress-activated, pro-apoptotic kinase which, following caspase-cleavage, enters the nucleus and induces chromatin condensation followed by internucleosomal DNA fragmentation (PubMed:11278283, PubMed:8566796, PubMed:8816758). Key component of the Hippo signaling pathway which plays a pivotal role in organ size control and tumor suppression by restricting proliferation and promoting apoptosis. The core of this pathwa

In [ ]:
# prompt: save formatted_prompts as name as "database_prompts.json"

with open("database_prompts.json", "w") as f:
  json.dump(formatted_prompts, f, indent=2)

print("formatted_prompts saved to database_prompts.json")


formatted_prompts saved to database_prompts.json


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from huggingface_hub import login

import os
login(token=os.environ["HF_TOKEN"])  # set HF_TOKEN as an environment variable / Colab secret

model_id = "elinas/Llama-3-13B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="auto"
)



Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/9.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/6.15G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/9.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

In [ ]:
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

Device set to use cuda:0


In [ ]:
print(json.dumps(formatted_prompts[12], indent=2))

[
  {
    "role": "system",
    "content": "You are a bioinformatics expert in kinase-substrate interactions. Your task is to evaluate whether the kinase phosphorylates the substrate, using only the evidence provided. If there is clear and direct evidence of phosphorylation, respond 'Yes'; otherwise, respond 'No'. Do not include any explanations or additional text, regardless of confidence."
  },
  {
    "role": "user",
    "content": "Protein A (Kinase): SRC\nProtein B (Substrate): Q9Y3Q4\n\nProtein A Description: Non-receptor protein tyrosine kinase which is activated following engagement of many different classes of cellular receptors including immune response receptors, integrins and other adhesion receptors, receptor protein tyrosine kinases, G protein-coupled receptors as well as cytokine receptors (PubMed:34234773). Participates in signaling pathways that control a diverse spectrum of biological activities including gene transcription, immune response, cell adhesion, cell cycle 

In [ ]:
messages = formatted_prompts[211]
# Convert messages to chat-style prompt using tokenizer's chat template
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Run inference
outputs = pipe(
    prompt,
    max_new_tokens=2,   # You mentioned you only need a short answer
    temperature=0.7,
    top_p=0.9,
    do_sample=False
)

# Print result
print(outputs[0]["generated_text"])


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a bioinformatics expert evaluating potential phosphorylation interactions between kinases and substrates. Based solely on the provided descriptions of both proteins, assess whether the kinase is likely to phosphorylate the substrate. If there is any evidence or functional relevance suggesting potential phosphorylation, respond with 'Yes'. Otherwise, respond with 'No'. Do not provide explanations. Only answer 'Yes' or 'No'.<|eot_id|><|start_header_id|>user<|end_header_id|>

Protein A (Kinase): DYRK3
Protein B (Substrate): Q9UKM9

Protein A Description: Dual-specificity protein kinase that promotes disassembly of several types of membraneless organelles during mitosis, such as stress granules, nuclear speckles and pericentriolar material (PubMed:29973724). Dual-specificity tyrosine-regulated kinases (DYRKs) autophosphorylate a critical tyrosine residue in their activation loop and phosphorylate their substrate on serine

In [ ]:
def extract_final_answer_binary(text):
    lines = [line.strip() for line in text.strip().split('\n') if line.strip()]

    for line in reversed(lines):
        if line == "Yes":
            return 1
        elif line == "No":
            return 0
    return -1

raw_output = outputs[0]["generated_text"]
final_answer = extract_final_answer_binary(raw_output)
print(final_answer)

1


In [ ]:
# Final
import pandas as pd
results=[]
for i, messages in enumerate(formatted_prompts):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    outputs = pipe(
        prompt,
        max_new_tokens=3,
        temperature=0.7,
        #max_new_tokens=10，
        #temperature=0.01,
        top_p=0.95,
        do_sample=False
    )

    raw_output = outputs[0]["generated_text"]
    final_answer = extract_final_answer_binary(raw_output)
    results.append({
        "index": i+2,
        "final_answer": final_answer,
    })

# save as csv
df = pd.DataFrame(results)
df.to_csv("llama_outputs2.csv", index=False)
print("Results saved to llama_outputs2.csv")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more de

Results saved to llama_outputs2.csv


In [ ]:
results

[{'index': 2, 'final_answer': 1},
 {'index': 3, 'final_answer': 1},
 {'index': 4, 'final_answer': 1},
 {'index': 5, 'final_answer': 1},
 {'index': 6, 'final_answer': 0},
 {'index': 7, 'final_answer': 0},
 {'index': 8, 'final_answer': 1},
 {'index': 9, 'final_answer': 1},
 {'index': 10, 'final_answer': 1},
 {'index': 11, 'final_answer': 1},
 {'index': 12, 'final_answer': 0},
 {'index': 13, 'final_answer': 0},
 {'index': 14, 'final_answer': 1},
 {'index': 15, 'final_answer': 0},
 {'index': 16, 'final_answer': 0},
 {'index': 17, 'final_answer': 1},
 {'index': 18, 'final_answer': 1},
 {'index': 19, 'final_answer': 1},
 {'index': 20, 'final_answer': 0},
 {'index': 21, 'final_answer': 1},
 {'index': 22, 'final_answer': 0},
 {'index': 23, 'final_answer': 1},
 {'index': 24, 'final_answer': 0}]

In [ ]:
# save as csv
df = pd.DataFrame(results)
df.to_csv("llama_outputs2.csv", index=False)
print("Results saved to llama_outputs2.csv")

Results saved to llama_outputs2.csv


In [ ]:
# prompt: 我只想截取output的回答部分 ，不要prompts ，只要结尾的Yes 或者 No

import pandas as pd
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from huggingface_hub import login
import re

# Preceding code to load data and create prompts
df = pd.read_excel('data kinase.xlsx')

with open('all_database_info.json', 'r') as f:
  database_info = json.load(f)

formatted_prompts = []
for i in range(2219) : # Or use a smaller range for testing
  row = df.iloc[i]
  uniprot_id = row['Uniprotid']
  kinase = row['kinase']
  # Add checks to ensure uniprot_id and kinase exist in database_info
  if uniprot_id in database_info and kinase in database_info:
      value_of_uniprot_id = database_info[uniprot_id]
      value_of_kinase = database_info[kinase]
      # Add checks to ensure 'Function' key exists
      if 'Function' in value_of_uniprot_id and 'Function' in value_of_kinase:
          prompt = [
                {
                    "role": "system",
                    "content": (
                        "You are a bioinformatics professor and preparing for an exam. "
                        "You are asked to answer questions related to gene phosphorylation and substrate-kinase interactions. "
                        "Please carefully read the provided references and base your answer solely on the information provided. "
                        "Only respond with 'Yes' or 'No'. Do not provide explanations or additional text, regardless of confidence."
                    )
                },
                {
                    "role": "user",
                    "content": (
                        f"Protein A (Kinase): {kinase}\n"
                        f"Protein B (Substrate): {uniprot_id}\n\n"
                        f"Protein A Description: {value_of_kinase['Function']}\n"
                        f"Protein B Description: {value_of_uniprot_id['Function']}\n"
                    )
                }
          ]
          formatted_prompts.append(prompt)
      else:
          print(f"Warning: 'Function' key missing for {uniprot_id} or {kinase}")
  else:
      print(f"Warning: Uniprot ID {uniprot_id} or Kinase {kinase} not found in database_info")


# Save prompts (optional, based on original code)
# with open("database_prompts.json", "w") as f:
#   json.dump(formatted_prompts, f, indent=2)

# print("formatted_prompts saved to database_prompts.json")

# Log in to Hugging Face (ensure you have the token)
# Replace with your actual token or set environment variable HF_TOKEN
import os
login(token=os.environ["HF_TOKEN"])  # set HF_TOKEN as an environment variable / Colab secret

model_id = "elinas/Llama-3-13B-Instruct"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="auto"
)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

# --- Code to process a single prompt and extract Yes/No ---

# Select a prompt to process (e.g., the first one)
if formatted_prompts: # Check if prompts list is not empty
    messages = formatted_prompts[0] # You can change the index

    # Convert messages to chat-style prompt
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Run inference
    outputs = pipe(
        prompt,
        max_new_tokens=5,   # Allow a few more tokens just in case, but still keep it short
        temperature=0.1,    # Use a lower temperature for less randomness
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id # Handle padding if needed
    )

    output_text = outputs[0]["generated_text"]

    # Extract the last occurrence of 'Yes' or 'No' from the output text
    # This regex looks for 'Yes' or 'No' at the end of the string or followed by non-word characters
    match = re.search(r"\b(Yes|No)\b(?:[^a-zA-Z0-9]*$)", output_text, re.IGNORECASE)

    answer = match.group(1) if match else "Unknown" # Default to "Unknown" if no match

    # Print only the extracted answer
    print(answer)

else:
    print("No prompts were generated.")


SyntaxError: invalid syntax (<ipython-input-76-3219352881>, line 1)